# LLM-project pipeline

Three-step flow: 1. RAG test → 2. Train and save → 3. Inference.
Run cells in order, or run only the step you need. Config is loaded from `qwen_math_flow/hyperparameters.py`; you can override variables in the next cell.

In [1]:
import json
import random
import re
import time
from pathlib import Path

import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from qwen_math_flow.download_model import download_qwen_2_15b
from qwen_math_flow.external_calculator import SafeEvalCalculatorClient, StubCalculatorClient
from qwen_math_flow.load_dataset import (
    build_deterministic_math_prompt,
    format_gsm8k_as_deterministic_json_chat,
    load_and_tokenize_math,
)
from qwen_math_flow.lora_finetune import create_lora_model, run_finetune
from qwen_math_flow.rag_calculator import RAGCalculatorLayer


c:\Users\Admin\Desktop\smu\LLM-project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. RAG test (no model)

In [ ]:
from qwen_math_flow.hyperparameters import USE_SAFE_EVAL_RAG_TEST

client = SafeEvalCalculatorClient() if USE_SAFE_EVAL_RAG_TEST else StubCalculatorClient()
rag = RAGCalculatorLayer(client)
tests = [
    "Result: [CALC: 2+3*4]",
    "First compute [CALC: 10/2], then add 5.",
]
print("RAG test (augment only, no model):")
for s in tests:
    out, pairs = rag.augment(s, inject_into_context=True)
    print(f"  in:  {s}")
    print(f"  out: {out}")
    print(f"  calc: {pairs}")
print("RAG test done.\n")

## 2. Train and save

In [3]:
from qwen_math_flow.hyperparameters import (
    ADAPTER_DIR,
    DATASET_NAME,
    DATASET_SPLIT,
    GRADIENT_ACCUMULATION_STEPS,
    LEARNING_RATE,
    LOAD_IN_4BIT,
    LORA_ALPHA,
    LORA_R,
    MAX_LENGTH,
    MAX_TRAIN_SAMPLES,
    MODEL_CACHE_DIR,
    NUM_EPOCHS,
    PER_DEVICE_TRAIN_BATCH_SIZE,
)

model, tokenizer = download_qwen_2_15b(
    cache_dir=MODEL_CACHE_DIR,
    load_in_4bit=LOAD_IN_4BIT,
    device_map="auto" if LOAD_IN_4BIT else None,
)
tokenized_train = load_and_tokenize_math(
    tokenizer,
    name=DATASET_NAME,
    split=DATASET_SPLIT,
    max_samples=MAX_TRAIN_SAMPLES,
    max_length=MAX_LENGTH,
    message_formatter=format_gsm8k_as_deterministic_json_chat,
)
cap_msg = "full train split" if MAX_TRAIN_SAMPLES is None else f"up to {MAX_TRAIN_SAMPLES} samples"
print(f"GSM8K training: {len(tokenized_train)} samples ({cap_msg}), {NUM_EPOCHS} epoch(s).")
peft_model = create_lora_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    use_4bit_or_8bit=LOAD_IN_4BIT,
)
run_finetune(
    peft_model,
    tokenizer,
    tokenized_train,
    output_dir=ADAPTER_DIR,
    num_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
)
print(f"Train done. Adapter + tokenizer saved to: {ADAPTER_DIR}\n")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Tokenizing (num_proc=1):   0%|          | 0/100 [00:00<?, ? examples/s]

GSM8K training: 100 samples (up to 100 samples), 2 epoch(s).


Step,Training Loss
10,1.442348


Train done. Adapter + tokenizer saved to: output/lora_math



## 3. Inference

In [ ]:
from qwen_math_flow.hyperparameters import (
    ADAPTER_DIR,
    LOAD_IN_4BIT_INFERENCE,
    MAX_NEW_TOKENS,
)


def extract_true_answer(answer: str) -> str:
    return answer.split("####")[-1].strip()


def extract_true_reasoning(answer: str) -> str:
    return answer.split("####")[0].strip()


def count_tokens(input_text: str, output_text: str) -> tuple[int, int]:
    input_tokens = len(tokenizer.encode(input_text))
    output_tokens = len(tokenizer.encode(output_text))
    return input_tokens, output_tokens


def extract_json(text: str) -> dict:
    matches = re.finditer(r"\{.*?\}", text, flags=re.DOTALL)
    for match in matches:
        try:
            parsed = json.loads(match.group())
            if "model_answer" not in parsed:
                parsed["model_answer"] = ""
            if "model_reasoning" not in parsed:
                parsed["model_reasoning"] = ""
            return parsed
        except json.JSONDecodeError:
            continue
    return {"model_answer": "", "model_reasoning": ""}


adapter_path = Path(ADAPTER_DIR)
with open(adapter_path / "adapter_config.json") as f:
    base_model_id = json.load(f).get("base_model_name_or_path", "Qwen/Qwen2-1.5B")

tokenizer = AutoTokenizer.from_pretrained(str(adapter_path), trust_remote_code=True)
base_kwargs = dict(device_map="auto", trust_remote_code=True)
if LOAD_IN_4BIT_INFERENCE:
    base_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
else:
    base_kwargs["torch_dtype"] = "auto"

base_model = AutoModelForCausalLM.from_pretrained(base_model_id, **base_kwargs)
model = PeftModel.from_pretrained(base_model, str(adapter_path))
model.eval()

dataset = load_dataset("gsm8k", "main", split="test")
dataset = dataset.filter(
    lambda x: len(x["answer"]) > 404
)
print(dataset)

results = []
start_time = time.time()

num_attempts = 2
consistent_count = 0


def generate_once(question: str, temperature: float = 0.7) -> tuple[str, str]:
    prompt = build_deterministic_math_prompt(question)
    messages = [{"role": "user", "content": prompt}]
    chat_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(chat_prompt, return_tensors="pt", truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    generation_kwargs = {
        "max_new_tokens": max(320, MAX_NEW_TOKENS),
        "return_dict_in_generate": False,
        "do_sample": True,
        "temperature": temperature,
        "repetition_penalty": 1.1,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    with torch.no_grad():
        out = model.generate(**inputs, **generation_kwargs)
    generated_tokens = out[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return prompt, response


for i, example in enumerate(dataset):
    question = example["question"]

    true_answer = extract_true_answer(example["answer"])
    true_reasoning = extract_true_reasoning(example["answer"])

    model_responses = []
    model_answers = []
    model_correct = []
    model_input_tokens = []
    model_output_tokens = []
    model_reasonings = []

    for attempt in range(num_attempts):
        prompt, response = generate_once(question, temperature=0.7)
        parsed = extract_json(response)

        model_answer = parsed.get("model_answer", "")
        model_reasoning = parsed.get("model_reasoning", "")

        is_correct = model_answer == true_answer
        input_tokens, output_tokens = count_tokens(prompt, response)

        model_answers.append(model_answer)
        model_correct.append(is_correct)
        model_input_tokens.append(input_tokens)
        model_output_tokens.append(output_tokens)
        model_reasonings.append(model_reasoning)
        model_responses.append(response)

    response_1 = model_responses[0]
    response_2 = model_responses[1]
    model_reasoning = model_reasonings[0]

    cleaned_answers = [ans for ans in model_answers]
    is_consistent = (
        len(set(cleaned_answers)) == 1
        and all(ans != "" for ans in cleaned_answers)
    )

    if is_consistent:
        consistent_count += 1

    avg_accuracy = sum(model_correct) / num_attempts
    avg_input_tokens = sum(model_input_tokens) / num_attempts
    avg_output_tokens = sum(model_output_tokens) / num_attempts

    results.append({
        "question": question,
        "true_reasoning": true_reasoning,
        "true_answer": true_answer,
        "model_reasoning": model_reasoning,
        "model_answers": model_answers,
        "model_correct": model_correct,
        "model_input_tokens": model_input_tokens,
        "model_output_tokens": model_output_tokens,
        "avg_accuracy": avg_accuracy,
        "avg_input_tokens": avg_input_tokens,
        "avg_output_tokens": avg_output_tokens,
        "consistent": is_consistent,
        "response_1": response_1,
        "response_2": response_2,
    })

end_time = time.time()

total_runtime = end_time - start_time
avg_runtime = total_runtime / len(results) if results else 0.0
print(f"Total runtime for {len(results)} examples: {total_runtime:.3f} sec")
print(f"Average runtime per example: {avg_runtime:.3f} sec")

accuracy = sum(r["avg_accuracy"] for r in results) / len(results) if results else 0.0
print("Final answer accuracy:", accuracy)

avg_input_tokens = sum(r["avg_input_tokens"] for r in results) / len(results) if results else 0.0
avg_output_tokens = sum(r["avg_output_tokens"] for r in results) / len(results) if results else 0.0

print("Avg input tokens:", avg_input_tokens)
print("Avg output tokens:", avg_output_tokens)

total_questions = len(results)
consistency_score = consistent_count / total_questions if total_questions > 0 else 0
print(f"Consistency score: {consistency_score:.4f}")

results_path = adapter_path / "inference_eval_results.json"
summary_path = adapter_path / "inference_eval_summary.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({
        "num_examples": len(results),
        "filter": "len(answer) > 404 on gsm8k test split",
        "num_attempts": num_attempts,
        "temperature": 0.7,
        "accuracy": accuracy,
        "avg_input_tokens": avg_input_tokens,
        "avg_output_tokens": avg_output_tokens,
        "consistency_score": consistency_score,
        "total_runtime_sec": total_runtime,
        "avg_runtime_sec": avg_runtime,
    }, f, indent=2)
print(f"Saved per-question results to: {results_path}")
print(f"Saved summary metrics to: {summary_path}")


Loading weights: 100%|██████████| 338/338 [00:03<00:00, 108.71it/s]


Dataset({
    features: ['question', 'answer'],
    num_rows: 245
})
